# Scraping — les 3 missions dans une seule cellule

Chaque mission **enrichit** le DataFrame de la précédente (pas de duplication) :
- **Mission 1** : titre + prix
- **Mission 2** : + catégorie + en stock
- **Mission 3** : + description + nombre exact en stock

Les missions sont reliées par la clé `id_livre` (l'identifiant unique de chaque
fiche, ex. `its-only-the-himalayas_981`).

⚠️ La Mission 3 visite les 1000 fiches une par une → comptez ~8 minutes.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL_SITE = "https://books.toscrape.com/"
TAUX_GBP_EUR = 1.17  # taux approximatif livre -> euro


# =====================================================================
# MISSION 1 : titre + prix  (DataFrame de base)
# =====================================================================
ids, titres, prix = [], [], []

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    soup = BeautifulSoup(requests.get(url).content, "html.parser")
    for bloc in soup.select("article.product_pod"):
        ids.append(bloc.h3.a["href"].split("/")[-2])          # clé unique
        titres.append(bloc.h3.a["title"])
        prix_texte = bloc.select_one("p.price_color").text
        prix.append(round(float(prix_texte.replace("£", "")) * TAUX_GBP_EUR, 2))

df = pd.DataFrame({"id_livre": ids, "titre": titres, "prix_euros": prix})


# =====================================================================
# MISSION 2 : + catégorie + en stock  (parcours des catégories)
# =====================================================================
soup = BeautifulSoup(requests.get(URL_SITE).content, "html.parser")
categories = {a.text.strip(): URL_SITE + a["href"]
              for a in soup.select("div.side_categories ul li ul li a")}

infos = []
for nom, url_categorie in categories.items():
    dossier = url_categorie.replace("index.html", "")
    url = url_categorie
    while url:
        soup = BeautifulSoup(requests.get(url).content, "html.parser")
        for bloc in soup.select("article.product_pod"):
            dispo = bloc.select_one("p.instock.availability").text
            infos.append({
                "id_livre": bloc.h3.a["href"].split("/")[-2],   # même clé
                "categorie": nom,
                "en_stock": "in stock" in dispo.lower(),
            })
        suivant = soup.select_one("li.next a")
        url = dossier + suivant["href"] if suivant else None

df = df.merge(pd.DataFrame(infos), on="id_livre")   # AJOUT des 2 colonnes


# =====================================================================
# MISSION 3 : + description + nombre exact en stock
#   (ces infos ne sont que sur la FICHE de chaque livre -> 1000 visites)
#   On visite les 1000 fiches une par une : c'est long, c'est normal.
# =====================================================================
cles, descriptions, nb_stocks = [], [], []

for numero, id_livre in enumerate(df["id_livre"]):
    url = f"https://books.toscrape.com/catalogue/{id_livre}/index.html"
    soup = BeautifulSoup(requests.get(url).content, "html.parser")

    # nombre en stock : "In stock (22 available)" -> 22
    dispo = soup.select_one("p.instock.availability").text
    nb_stocks.append(int(dispo.split("(")[1].split(" ")[0]) if "(" in dispo else 0)

    # description : paragraphe juste après le titre "Product Description" (parfois absent)
    bloc_desc = soup.select_one("#product_description ~ p")
    descriptions.append(bloc_desc.text.strip() if bloc_desc else "")

    cles.append(id_livre)

    # petit suivi pour voir que ça avance (tous les 100 livres)
    if numero % 100 == 0:
        print(numero, "livres traités...")

df_details = pd.DataFrame({"id_livre": cles, "description": descriptions, "nb_stock": nb_stocks})
df = df.merge(df_details, on="id_livre")   # AJOUT des 2 colonnes


# ---------------------------------------------------------------------
# Résultat final
# ---------------------------------------------------------------------
df = df.drop(columns="id_livre")           # la clé n'est plus utile à l'affichage
print(df.shape)
df.head()

0 livres traités...


100 livres traités...


200 livres traités...


300 livres traités...


400 livres traités...


...

700 livres traités...


800 livres traités...


900 livres traités...


(1000, 6)


,titre,prix_euros,categorie,en_stock,description,nb_stock
0,A Light in the Attic,60.57,Poetry,True,It's hard to imagine a world without A Light i...,22
1,Tipping the Velvet,62.88,Historical Fiction,True,"""Erotic and absorbing...Written with starling ...",20
2,Soumission,58.62,Fiction,True,"Dans une France assez proche de la nôtre, un h...",20
3,Sharp Objects,55.95,Mystery,True,"WICKED above her hipbone, GIRL across her hear...",20
4,Sapiens: A Brief History of Humankind,63.45,History,True,From a renowned historian comes a groundbreaki...,20


In [23]:
import re

texte = "Contact : alice@example.fr ou 06 12 34 56 78"

# 1. search — premier match
m = re.search(r"\d{2} \d{2} \d{2} \d{2} \d{2}", texte)
if m:
    print(m.group())        # "06 12 34 56 78"
    print(m.start(), m.end())  # 30, 44


# Démonstration
text = """
Bonjour Alice (alice.dupont@example.fr),
Votre commande au 06 12 34 56 78 livrée 75001 Paris.
IBAN : FR76 3000 4000 0312 3456 7890 143
"""


m = re.search(r"(\d{2})/(\d{2})/(\d{4})", "Né le 03/07/1995")
print(m.group(0))       # match complet : "03/07/1995"
print(f"{m.group(1)}/{m.group(2)}/{m.group(3)}")    # "03", "07", "1995"
print(m.groups())

06 12 34 56 78
30 44
03/07/1995
03/07/1995
('03', '07', '1995')


In [27]:
s = "Né le 03/07/1995"
m = re.search(r"(?P<jour>\d{2})/(?P<mois>\d{2})/(?P<annee>\d{4})", s)
m.group("annee")           # "1995"
m.groupdict()

{'jour': '03', 'mois': '07', 'annee': '1995'}

In [31]:
import urllib.robotparser as rp
r = rp.RobotFileParser()
r.set_url("https://books.toscrape.com/index.html/robots.txt"); r.read()

URLError: <urlopen error [Errno -3] Temporary failure in name resolution>